In [2]:
import numpy as np
import pandas as pd
import joblib
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_STATE = 42
TIMESTEPS = 1  # treat each row as a single timestep; increase if data has temporal structure
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ── data ─────────────────────────────────────────────────────────────────────
cb_train = pd.read_parquet("datasets/train")
cb_val   = pd.read_parquet("datasets/test_all_features.parquet")

feat_cols = [c for c in cb_train.columns
             if c not in ("user_id", "target")
             and not pd.api.types.is_datetime64_any_dtype(cb_train[c])]

print(f"  LSTM  — train: {cb_train.shape[0]} rows, {len(feat_cols)} features")

X_tr_raw = cb_train[feat_cols].values.astype(np.float32)
y_tr_raw = np.log1p(cb_train["target"].values.astype(np.float32))

X_val_raw    = cb_val[feat_cols].values.astype(np.float32)
y_val_raw    = np.log1p(cb_val["target"].values.astype(np.float32))
y_val_target = cb_val["target"].values  # raw, for RMSLE scoring

# reshape to (N, TIMESTEPS, features)
X_tr  = X_tr_raw.reshape(-1, TIMESTEPS, X_tr_raw.shape[1] // TIMESTEPS)
X_val = X_val_raw.reshape(-1, TIMESTEPS, X_val_raw.shape[1] // TIMESTEPS)
N_FEATURES = X_tr.shape[2]

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

# ── model ─────────────────────────────────────────────────────────────────────
class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)        # (batch, seq, hidden)
        out = out[:, -1, :]          # last timestep
        return self.head(out).squeeze(-1)


# ── training helper ───────────────────────────────────────────────────────────
def train_lstm(params, X_tr, y_tr, X_val, y_val, max_epochs=100, patience=10):
    tr_ds  = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
    tr_dl  = DataLoader(tr_ds,  batch_size=params["batch_size"], shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=2048, shuffle=False)

    model = LSTMRegressor(
        n_features=N_FEATURES,
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"],
                                 weight_decay=params["weight_decay"])
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state    = None
    no_improve    = 0
    best_epoch    = 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_loss += criterion(model(xb), yb).item() * len(xb)
        val_loss /= len(val_ds)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch    = epoch
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_epoch


def predict(model, X):
    model.eval()
    dl = DataLoader(TensorDataset(torch.tensor(X)), batch_size=2048, shuffle=False)
    preds = []
    with torch.no_grad():
        for (xb,) in dl:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(preds)


# ── optuna objective ──────────────────────────────────────────────────────────
print("\nOptuna tuning LSTM (20 trials)...")


def lstm_objective(trial):
    params = dict(
        hidden_size  = trial.suggest_categorical("hidden_size", [64, 128, 256, 512]),
        num_layers   = trial.suggest_int("num_layers", 1, 3),
        dropout      = trial.suggest_float("dropout", 0.0, 0.5),
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        batch_size   = trial.suggest_categorical("batch_size", [256, 512, 1024]),
    )
    model, _ = train_lstm(params, X_tr, y_tr_raw, X_val, y_val_raw,
                          max_epochs=80, patience=10)
    score = rmsle(y_val_target, np.expm1(predict(model, X_val)))
    return score


study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(lstm_objective, n_trials=20)
best_params = study.best_params

print(f"  Best LSTM RMSLE: {study.best_value:.5f}")
print(f"  Params: {best_params}")

# ── final model on split (for validation predictions) ─────────────────────────
print("\nTraining final LSTM on train split...")
final_model_split, best_epoch = train_lstm(
    best_params, X_tr, y_tr_raw, X_val, y_val_raw,
    max_epochs=200, patience=15,
)
split_rmsle = rmsle(y_val_target, np.expm1(predict(final_model_split, X_val)))
print(f"  Final split RMSLE: {split_rmsle:.5f}  (best epoch: {best_epoch})")

# ── retrain on full data using best_epoch iterations ──────────────────────────
print("\nTraining final LSTM on full data (train + val)...")
X_full = np.concatenate([X_tr, X_val], axis=0)
y_full = np.concatenate([y_tr_raw, y_val_raw], axis=0)

# use best_epoch+1 as fixed epoch count, no early stopping
full_params = {**best_params}
full_ds = TensorDataset(torch.tensor(X_full), torch.tensor(y_full))
full_dl = DataLoader(full_ds, batch_size=full_params["batch_size"], shuffle=True)

final_model_full = LSTMRegressor(
    n_features=N_FEATURES,
    hidden_size=full_params["hidden_size"],
    num_layers=full_params["num_layers"],
    dropout=full_params["dropout"],
).to(DEVICE)

optimizer_full = torch.optim.Adam(
    final_model_full.parameters(),
    lr=full_params["lr"],
    weight_decay=full_params["weight_decay"],
)
criterion = nn.MSELoss()

for epoch in range(best_epoch + 1):
    final_model_full.train()
    for xb, yb in full_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer_full.zero_grad()
        loss = criterion(final_model_full(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(final_model_full.parameters(), 1.0)
        optimizer_full.step()

print(f"  Full model trained for {best_epoch + 1} epochs.")

# ── save ──────────────────────────────────────────────────────────────────────
torch.save(final_model_split.state_dict(), "model/lstm_split1.pt")
torch.save(final_model_full.state_dict(),  "model/lstm_full.pt")
joblib.dump({"feat_cols": feat_cols, "best_params": best_params,
             "best_epoch": best_epoch, "n_features": N_FEATURES,
             "timesteps": TIMESTEPS},
            "model/lstm_meta.joblib")


KeyboardInterrupt: 